# Notebook 1 — Data Preparation

## Pharmacy Monthly Demand Forecasting Pipeline

**Objective:** Load raw transaction and stock data, convert dates, sort by drug and date, and aggregate sales to **monthly demand** per drug. Demand is derived from `quantity_dispensed` in transaction data—no assumptions.

**Output:** Clean dataset `outputs/monthly_demand.csv` with columns: `drug_id`, `month`, `monthly_demand`.

## 1. Imports and Paths

In [1]:
import pandas as pd
from pathlib import Path

# Project paths (run from project root, pharmacy_forecasting/, or notebooks/)
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "pharmacy_forecasting":
    PROJECT_ROOT = PROJECT_ROOT / "pharmacy_forecasting"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Datasets

In [2]:
sales = pd.read_csv(DATA_DIR / "sales_transactions.csv")
stock = pd.read_csv(DATA_DIR / "stock_receipts.csv")
print("Sales shape:", sales.shape)
print("Stock shape:", stock.shape)
sales.head()

Sales shape: (223389, 8)
Stock shape: (340, 8)


,transaction_id,transaction_date,drug_id,drug_name,category,quantity_dispensed,unit_price,facility_id
0,T1,2022-01-01,D001,Artemether-Lumefantrine,Antimalarial,2,12.02,F001
1,T2,2022-01-01,D001,Artemether-Lumefantrine,Antimalarial,3,25.67,F001
2,T3,2022-01-01,D001,Artemether-Lumefantrine,Antimalarial,5,32.05,F001
3,T4,2022-01-01,D001,Artemether-Lumefantrine,Antimalarial,3,5.93,F001
4,T5,2022-01-01,D001,Artemether-Lumefantrine,Antimalarial,2,37.49,F001


## 3. Convert Date Columns and Sort

In [3]:
sales["transaction_date"] = pd.to_datetime(sales["transaction_date"])
stock["stock_received_date"] = pd.to_datetime(stock["stock_received_date"])
sales = sales.sort_values(["drug_id", "transaction_date"]).reset_index(drop=True)
stock = stock.sort_values(["drug_id", "stock_received_date"]).reset_index(drop=True)
sales["transaction_date"].min(), sales["transaction_date"].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

## 4. Aggregate to Monthly Demand

Demand is **derived** from transaction data: sum of `quantity_dispensed` per drug per month (freq=`"MS"` = month start).

In [10]:
monthly_demand = (
    sales.groupby(["drug_id", pd.Grouper(key="transaction_date", freq="MS")])["quantity_dispensed"]
    .sum()
    .reset_index()
)
monthly_demand = monthly_demand.rename(
    columns={
        "transaction_date": "month",
        "quantity_dispensed": "monthly_demand"
    }
)

monthly_demand.head(15)

,drug_id,month,monthly_demand
0,D001,2022-01-01,2216
1,D001,2022-02-01,1998
2,D001,2022-03-01,2015
3,D001,2022-04-01,3171
4,D001,2022-05-01,2914
5,D001,2022-06-01,1909
6,D001,2022-07-01,2086
7,D001,2022-08-01,1391
8,D001,2022-09-01,2089
9,D001,2022-10-01,3302


## 5. Ensure Continuous Monthly Index per Drug

Fill missing months with 0 (e.g. stockout or no transactions).

In [5]:
def ensure_continuous_months(df: pd.DataFrame, date_col: str = "month", value_col: str = "monthly_demand") -> pd.DataFrame:
    """For each drug_id, create a full range of months and fill missing with 0."""
    all_months = pd.date_range(df[date_col].min(), df[date_col].max(), freq="MS")
    drug_ids = df["drug_id"].unique()
    rows = []
    for did in drug_ids:
        sub = df[df["drug_id"] == did].set_index(date_col).reindex(all_months, fill_value=0)
        sub["drug_id"] = did
        sub = sub.reset_index().rename(columns={"index": date_col})
        sub[value_col] = sub[value_col].fillna(0).astype(int)
        rows.append(sub)
    out = pd.concat(rows, ignore_index=True)
    return out[["drug_id", date_col, value_col]]

In [11]:
monthly_demand = ensure_continuous_months(monthly_demand)
monthly_demand = monthly_demand.sort_values(["drug_id", "month"]).reset_index(drop=True)
monthly_demand.head(20)

,drug_id,month,monthly_demand
0,D001,2022-01-01,2216
1,D001,2022-02-01,1998
2,D001,2022-03-01,2015
3,D001,2022-04-01,3171
4,D001,2022-05-01,2914
5,D001,2022-06-01,1909
6,D001,2022-07-01,2086
7,D001,2022-08-01,1391
8,D001,2022-09-01,2089
9,D001,2022-10-01,3302


## 6. Save Clean Monthly Dataset

In [12]:
out_path = OUTPUTS_DIR / "monthly_demand.csv"
monthly_demand.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print("Shape:", monthly_demand.shape)
print("Columns:", list(monthly_demand.columns))
monthly_demand.tail(10)

Saved: C:\Users\hp\Desktop\prototype2\pharmacy_forecasting\outputs\monthly_demand.csv
Shape: (360, 3)
Columns: ['drug_id', 'month', 'monthly_demand']


,drug_id,month,monthly_demand
350,D010,2024-03-01,2117
351,D010,2024-04-01,1772
352,D010,2024-05-01,1199
353,D010,2024-06-01,1039
354,D010,2024-07-01,1750
355,D010,2024-08-01,998
356,D010,2024-09-01,1989
357,D010,2024-10-01,1432
358,D010,2024-11-01,967
359,D010,2024-12-01,2081


## 7. Inventory balance — opening stock, receipts, and dispensing

In addition to monthly demand, we compute a **per-drug stock balance** using:

\[
\text{current\_stock} = \text{opening\_stock\_units} + \text{total\_received} - \text{total\_dispensed}
\]

Inputs:
- `data/opening_stock.csv`
- `data/stock_receipts.csv`
- `data/sales_transactions.csv`

Output: `outputs/stock_status.csv` for use in the dashboard.

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

# Load opening stock, receipts, and sales
opening_stock = pd.read_csv(DATA_DIR / "opening_stock.csv")
stock_raw = pd.read_csv(DATA_DIR / "stock_receipts.csv")
sales_raw = pd.read_csv(DATA_DIR / "sales_transactions.csv")

print("Opening stock shape:", opening_stock.shape)
print("Stock receipts shape:", stock_raw.shape)
print("Sales shape:", sales_raw.shape)
opening_stock.head()

Opening stock shape: (10, 3)
Stock receipts shape: (340, 8)
Sales shape: (223389, 8)


,drug_id,drug_name,opening_stock_units
0,D001,Artemether-Lumefantrine,2860
1,D002,Amoxicillin 500mg,2563
2,D003,Paracetamol 500mg,4033
3,D004,Ceftriaxone Injection,2769
4,D005,Metformin 850mg,4875


In [4]:
# Standardise dtypes and keys
opening_stock["drug_id"] = opening_stock["drug_id"].astype(str)
opening_stock["opening_stock_units"] = pd.to_numeric(opening_stock["opening_stock_units"], errors="coerce").fillna(0).astype(int)

stock_raw["drug_id"] = stock_raw["drug_id"].astype(str)
stock_raw["quantity_received"] = pd.to_numeric(stock_raw["quantity_received"], errors="coerce").fillna(0).astype(int)

sales_raw["drug_id"] = sales_raw["drug_id"].astype(str)
sales_raw["quantity_dispensed"] = pd.to_numeric(sales_raw["quantity_dispensed"], errors="coerce").fillna(0).astype(int)

In [5]:
# Aggregate totals per drug
total_received = (
    stock_raw.groupby(["drug_id", "drug_name"], as_index=False)["quantity_received"]
    .sum()
    .rename(columns={"quantity_received": "total_received"})
)

total_dispensed = (
    sales_raw.groupby(["drug_id", "drug_name"], as_index=False)["quantity_dispensed"]
    .sum()
    .rename(columns={"quantity_dispensed": "total_dispensed"})
)

# Merge opening stock with totals
stock_status = opening_stock.merge(total_received, on=["drug_id", "drug_name"], how="left")
stock_status = stock_status.merge(total_dispensed, on=["drug_id", "drug_name"], how="left")

stock_status["total_received"] = stock_status["total_received"].fillna(0).astype(int)
stock_status["total_dispensed"] = stock_status["total_dispensed"].fillna(0).astype(int)

stock_status["current_stock"] = (
    stock_status["opening_stock_units"].astype(int)
    + stock_status["total_received"].astype(int)
    - stock_status["total_dispensed"].astype(int)
)
stock_status["current_stock"] = stock_status["current_stock"].clip(lower=0)

stock_status

,drug_id,drug_name,opening_stock_units,total_received,total_dispensed,current_stock
0,D001,Artemether-Lumefantrine,2860,66969,69379,450
1,D002,Amoxicillin 500mg,2563,52553,53993,1123
2,D003,Paracetamol 500mg,4033,65270,67565,1738
3,D004,Ceftriaxone Injection,2769,64493,65173,2089
4,D005,Metformin 850mg,4875,65882,68914,1843
5,D006,Salbutamol Inhaler,2791,66657,68882,566
6,D007,ORS Sachets,2553,67761,69622,692
7,D008,Coartem Pediatric,2936,63996,66147,785
8,D009,Azithromycin 250mg,4694,62519,64837,2376
9,D010,Ibuprofen 400mg,4335,62344,64545,2134


In [6]:
# Basic validation: preview and summary
print("Stock status shape:", stock_status.shape)
print(stock_status.head())
print("\nCurrent stock summary:")
print(stock_status["current_stock"].describe())

Stock status shape: (10, 6)
  drug_id                drug_name  opening_stock_units  total_received  \
0    D001  Artemether-Lumefantrine                 2860           66969   
1    D002        Amoxicillin 500mg                 2563           52553   
2    D003        Paracetamol 500mg                 4033           65270   
3    D004    Ceftriaxone Injection                 2769           64493   
4    D005          Metformin 850mg                 4875           65882   

   total_dispensed  current_stock  
0            69379            450  
1            53993           1123  
2            67565           1738  
3            65173           2089  
4            68914           1843  

Current stock summary:
count      10.000000
mean     1379.600000
std       732.321909
min       450.000000
25%       715.250000
50%      1430.500000
75%      2027.500000
max      2376.000000
Name: current_stock, dtype: float64


In [9]:
from pathlib import Path

OUTPUTS_DIR = Path("../outputs")

# Save processed stock status for dashboard use
stock_out_path = OUTPUTS_DIR / "stock_status.csv"
stock_status.to_csv(stock_out_path, index=False)
print(f"Saved: {stock_out_path}")
stock_status

Saved: ..\outputs\stock_status.csv


,drug_id,drug_name,opening_stock_units,total_received,total_dispensed,current_stock
0,D001,Artemether-Lumefantrine,2860,66969,69379,450
1,D002,Amoxicillin 500mg,2563,52553,53993,1123
2,D003,Paracetamol 500mg,4033,65270,67565,1738
3,D004,Ceftriaxone Injection,2769,64493,65173,2089
4,D005,Metformin 850mg,4875,65882,68914,1843
5,D006,Salbutamol Inhaler,2791,66657,68882,566
6,D007,ORS Sachets,2553,67761,69622,692
7,D008,Coartem Pediatric,2936,63996,66147,785
8,D009,Azithromycin 250mg,4694,62519,64837,2376
9,D010,Ibuprofen 400mg,4335,62344,64545,2134
